# Datathon 2026 – Vòng 1: Đáp án 10 Câu Trắc Nghiệm

## Phân tích đề trước khi tính

| Câu | Tính gì | Bảng chính | Điều kiện / trick tiềm ẩn |
|-----|---------|------------|---------------------------|
| Q1 | Trung vị inter-order gap (ngày) | orders | Chỉ lấy KH có >1 đơn; gap tính per-customer consecutive |
| Q2 | Segment có avg gross margin cao nhất | products | Công thức (price−cogs)/price, average per product rồi group |
| Q3 | Lý do trả hàng phổ biến nhất cho Streetwear | returns + products | Join theo product_id, filter category |
| Q4 | Traffic source có avg bounce_rate thấp nhất | web_traffic | Avg trên toàn bộ ngày xuất hiện của source đó |
| Q5 | % dòng order_items có promo_id not null | order_items | Đơn giản; promo_id_2 không tính |
| Q6 | Age group có avg đơn/khách cao nhất | customers + orders | Chỉ KH có age_group not null; avg = tổng đơn / số KH trong nhóm |
| Q7 | Region có tổng revenue cao nhất | orders + order_items + geography | **Trick**: sales.csv không có region → phải tính từ transactional tables |
| Q8 | Payment method phổ biến nhất cho đơn cancelled | orders | Filter order_status = 'cancelled' |
| Q9 | Size (S/M/L/XL) có return rate cao nhất | returns + order_items + products | Rate = count(returns) / count(order_items) per size |
| Q10 | Kỳ trả góp có avg payment_value cao nhất | payments | Group by installments, lấy avg payment_value |

## Setup

In [1]:
import pandas as pd
import numpy as np

# Load all tables từ thư mục 'input'
orders      = pd.read_csv('input/orders.csv', parse_dates=['order_date'])
order_items = pd.read_csv('input/order_items.csv', low_memory=False)
products    = pd.read_csv('input/products.csv')
customers   = pd.read_csv('input/customers.csv')
returns     = pd.read_csv('input/returns.csv')
web_traffic = pd.read_csv('input/web_traffic.csv')
payments    = pd.read_csv('input/payments.csv')
geography   = pd.read_csv('input/geography.csv')

print(f"orders: {orders.shape}, order_items: {order_items.shape}")
print(f"products: {products.shape}, customers: {customers.shape}")
print(f"returns: {returns.shape}, payments: {payments.shape}")

orders: (646945, 8), order_items: (714669, 7)
products: (2412, 8), customers: (121930, 7)
returns: (39939, 7), payments: (646945, 4)


---
## Q1 — Trung vị inter-order gap
**Câu hỏi:** Trong số các khách hàng có nhiều hơn một đơn hàng, trung vị số ngày giữa hai lần mua liên tiếp là bao nhiêu?

**Logic:** Sort orders theo (customer_id, order_date) → tính shift(1) → lấy gap → median trên tất cả các gap (chỉ các dòng có gap, tức KH có ≥2 đơn).

In [2]:
cust_orders = orders.sort_values(['customer_id', 'order_date'])
cust_orders['prev_date'] = cust_orders.groupby('customer_id')['order_date'].shift(1)
cust_orders['gap_days'] = (cust_orders['order_date'] - cust_orders['prev_date']).dt.days

# Chỉ lấy các gap hợp lệ (KH có >1 đơn)
gaps = cust_orders[cust_orders['gap_days'].notna()]['gap_days']

print(f"Số KH có >1 đơn: {cust_orders[cust_orders['gap_days'].notna()]['customer_id'].nunique():,}")
print(f"Tổng số gap records: {len(gaps):,}")
print(f"Median gap: {gaps.median():.1f} ngày")
print(f"Mean gap:   {gaps.mean():.1f} ngày")
print()
print("✅ ĐÁP ÁN Q1: C) 144 ngày")

Số KH có >1 đơn: 67,888
Tổng số gap records: 556,699
Median gap: 144.0 ngày
Mean gap:   285.6 ngày

✅ ĐÁP ÁN Q1: C) 144 ngày


---
## Q2 — Segment có gross margin trung bình cao nhất
**Công thức:** `margin = (price - cogs) / price` tính per product, rồi avg theo segment.

In [3]:
products['gross_margin'] = (products['price'] - products['cogs']) / products['price']

seg_margin = (
    products.groupby('segment')['gross_margin']
    .mean()
    .sort_values(ascending=False)
    .rename('avg_gross_margin')
)
print(seg_margin.apply(lambda x: f"{x:.4f} ({x*100:.2f}%)"))
print()
print("✅ ĐÁP ÁN Q2: D) Standard (31.3%)")

segment
Standard       0.3134 (31.34%)
Premium        0.2854 (28.54%)
All-weather    0.2842 (28.42%)
Activewear     0.2656 (26.56%)
Performance    0.2636 (26.36%)
Balanced       0.2580 (25.80%)
Trendy         0.2408 (24.08%)
Everyday       0.2363 (23.63%)
Name: avg_gross_margin, dtype: object

✅ ĐÁP ÁN Q2: D) Standard (31.3%)


---
## Q3 — Lý do trả hàng phổ biến nhất cho Streetwear
**Logic:** Filter products WHERE category = 'Streetwear' → merge với returns → value_counts return_reason.

In [4]:
sw_product_ids = products.loc[products['category'] == 'Streetwear', 'product_id']
sw_returns = returns[returns['product_id'].isin(sw_product_ids)]

print(f"Số bản ghi trả hàng Streetwear: {len(sw_returns):,}")
print()
print(sw_returns['return_reason'].value_counts())
print()
print("✅ ĐÁP ÁN Q3: B) wrong_size")

Số bản ghi trả hàng Streetwear: 21,799

return_reason
wrong_size          7626
defective           4330
not_as_described    3854
changed_mind        3830
late_delivery       2159
Name: count, dtype: int64

✅ ĐÁP ÁN Q3: B) wrong_size


---
## Q4 — Traffic source có bounce_rate trung bình thấp nhất
**Lưu ý:** Các giá trị bounce_rate rất sát nhau (~0.0045), nhưng email_campaign dẫn đầu.

In [5]:
br_by_source = (
    web_traffic.groupby('traffic_source')['bounce_rate']
    .mean()
    .sort_values()
)
print(br_by_source.apply(lambda x: f"{x:.6f}"))
print()
print("✅ ĐÁP ÁN Q4: C) email_campaign")

traffic_source
email_campaign    0.004458
social_media      0.004476
paid_search       0.004478
referral          0.004499
organic_search    0.004504
direct            0.004511
Name: bounce_rate, dtype: object

✅ ĐÁP ÁN Q4: C) email_campaign


---
## Q5 — % dòng order_items có promo_id not null
**Lưu ý:** Chỉ tính `promo_id` (cột 1), không tính `promo_id_2`.

In [6]:
total_rows   = len(order_items)
promo_rows   = order_items['promo_id'].notna().sum()
pct_with_promo = promo_rows / total_rows * 100

print(f"Tổng dòng order_items: {total_rows:,}")
print(f"Dòng có promo_id:       {promo_rows:,}")
print(f"Tỷ lệ:                  {pct_with_promo:.1f}%")
print()
print("✅ ĐÁP ÁN Q5: C) 39%")

Tổng dòng order_items: 714,669
Dòng có promo_id:       276,316
Tỷ lệ:                  38.7%

✅ ĐÁP ÁN Q5: C) 39%


---
## Q6 — Age group có avg đơn/khách cao nhất
**Logic:** Chỉ xét KH có age_group not null. Avg = tổng số đơn của nhóm / số KH trong nhóm.

**Lưu ý:** KH không có đơn hàng nào sẽ có order_count = 0 (left join).

In [7]:
cust_order_count = orders.groupby('customer_id').size().reset_index(name='order_count')

cust_with_age = customers[customers['age_group'].notna()][['customer_id', 'age_group']]
merged = cust_with_age.merge(cust_order_count, on='customer_id', how='left')
merged['order_count'] = merged['order_count'].fillna(0)

avg_orders = (
    merged.groupby('age_group')['order_count']
    .mean()
    .sort_values(ascending=False)
)
print(avg_orders.apply(lambda x: f"{x:.4f}"))
print()
print("✅ ĐÁP ÁN Q6: A) 55+")

age_group
55+      5.4069
45-54    5.3572
35-44    5.3373
25-34    5.2452
18-24    5.2267
Name: order_count, dtype: object

✅ ĐÁP ÁN Q6: A) 55+


---
## Q7 — Region có tổng doanh thu cao nhất
**Trick quan trọng:** `sales.csv` chỉ có Date/Revenue/COGS tổng hợp — không có thông tin địa lý.

Phải tính revenue từ `order_items` (quantity × unit_price) → join `orders` (để lấy zip) → join `geography` (để lấy region).

In [8]:
# Revenue per order = sum(quantity * unit_price)
order_revenue = (
    order_items.assign(line_revenue=order_items['quantity'] * order_items['unit_price'])
    .groupby('order_id')['line_revenue'].sum()
    .reset_index()
)

orders_with_region = (
    orders[['order_id', 'zip']]
    .merge(geography[['zip', 'region']], on='zip', how='left')
    .merge(order_revenue, on='order_id', how='left')
)

region_revenue = (
    orders_with_region.groupby('region')['line_revenue']
    .sum()
    .sort_values(ascending=False)
)
print(region_revenue.apply(lambda x: f"{x/1e9:.3f}B VND"))
print(f"\nEast/Central ratio: {region_revenue['East']/region_revenue['Central']:.2f}x")
print()
print("✅ ĐÁP ÁN Q7: C) East")

region
East       7.638B VND
Central    4.942B VND
West       3.851B VND
Name: line_revenue, dtype: object

East/Central ratio: 1.55x

✅ ĐÁP ÁN Q7: C) East


---
## Q8 — Payment method phổ biến nhất cho đơn cancelled

In [9]:
cancelled_orders = orders[orders['order_status'] == 'cancelled']

print(f"Tổng đơn cancelled: {len(cancelled_orders):,}")
print()
print(cancelled_orders['payment_method'].value_counts())
print()
print("✅ ĐÁP ÁN Q8: A) credit_card")

Tổng đơn cancelled: 59,462

payment_method
credit_card      28452
cod              15468
paypal            7817
apple_pay         5190
bank_transfer     2535
Name: count, dtype: int64

✅ ĐÁP ÁN Q8: A) credit_card


---
## Q9 — Size có return rate cao nhất
**Công thức:** `rate(size) = count(returns records cho size đó) / count(order_items rows cho size đó)`

Cả tử và mẫu đều join với products để lấy size.

In [10]:
sizes_of_interest = ['S', 'M', 'L', 'XL']

# Order items per size
oi_with_size = order_items.merge(products[['product_id', 'size']], on='product_id', how='left')
oi_counts = oi_with_size[oi_with_size['size'].isin(sizes_of_interest)]['size'].value_counts()

# Returns per size
ret_with_size = returns.merge(products[['product_id', 'size']], on='product_id', how='left')
ret_counts = ret_with_size[ret_with_size['size'].isin(sizes_of_interest)]['size'].value_counts()

return_rate = (ret_counts / oi_counts).sort_values(ascending=False)

summary = pd.DataFrame({'order_items': oi_counts, 'returns': ret_counts, 'return_rate': return_rate})
print(summary)
print()
print("✅ ĐÁP ÁN Q9: A) S")

      order_items  returns  return_rate
size                                   
L          173174     9741     0.056250
M          176428     9820     0.055660
S          172042     9723     0.056515
XL         193025    10655     0.055200

✅ ĐÁP ÁN Q9: A) S


---
## Q10 — Kỳ trả góp có avg payment_value cao nhất
**Lưu ý:** Installment = 2 là outlier đặc biệt (avg rất thấp ~708), có thể là kế hoạch đặc biệt.

In [11]:
avg_pv = (
    payments.groupby('installments')['payment_value']
    .agg(['mean', 'count'])
    .rename(columns={'mean': 'avg_payment_value', 'count': 'num_orders'})
    .sort_values('avg_payment_value', ascending=False)
)
print(avg_pv)
print()
print("✅ ĐÁP ÁN Q10: C) 6 kỳ")

              avg_payment_value  num_orders
installments                               
6                  24446.654403      109910
3                  24399.635486      218949
12                 24245.772694       54126
1                  24113.274166      262866
2                    708.473729        1094

✅ ĐÁP ÁN Q10: C) 6 kỳ


---
## Tổng hợp đáp án

| Câu | Đáp án | Giá trị cụ thể |
|-----|--------|----------------|
| Q1  | **C) 144 ngày** | Median inter-order gap = 144.0 ngày |
| Q2  | **D) Standard** | Avg gross margin = 31.3% (cao nhất) |
| Q3  | **B) wrong_size** | 7,626 bản ghi (>defective 4,330) |
| Q4  | **C) email_campaign** | Avg bounce_rate = 0.00446 (thấp nhất) |
| Q5  | **C) 39%** | 276,316 / 714,669 = 38.7% ≈ 39% |
| Q6  | **A) 55+** | Avg 5.41 đơn/khách (cao nhất) |
| Q7  | **C) East** | Revenue ~7.64B (gấp 1.55x Central, 1.98x West) |
| Q8  | **A) credit_card** | 28,452 / 59,462 đơn cancelled (47.8%) |
| Q9  | **A) S** | Return rate 5.65% (S > L > M > XL) |
| Q10 | **C) 6 kỳ** | Avg payment_value = 24,447 (cao nhất) |